# B3 — mixed displacement-stress formulation: RECALIBRATED re-run (attempt 2)

**Follow-up paper, Professor Rabczuk's item 1.** Attempt 1 (50,000 iterations, `p_output_scale=500.0`, saved to `b3_mixed_stress_training/`) **completed but failed**: combined displacement/energy/reaction errors against the held-out set were ~100x worse than the DEM baseline (energy 181.9%, reaction 216.9% vs. the baseline's 0.3%/3.0%), and the `constitutive` loss term showed no clear downward trend across the whole run. One real, measured problem was found and fixed since: `calibrate_p_output_scale.py` measured the untrained network's real raw stress-channel output against a realistic true-P magnitude (built from real sampled E/nu/phi) and derived a scale of ~246.6, vs. the 500.0 actually used — a real ~2x over-estimate. `P_OUTPUT_SCALE` is now 250.0.

**Important: this run writes to a NEW output directory** (`b3_mixed_stress_training_v2`), not the old one — resuming the old, wrong-scale checkpoint under the new scale would silently mix two inconsistent trainings (the code now actively refuses to do this if you ever point it at a mismatched checkpoint).

**Honest caveat, not a promise this fixes it**: a 2x scale correction is a real improvement, but much smaller than the kind of mismatch (5-50x) that explained `train_B3.py`'s own original `OUTPUT_SCALE` failure, and the first run's own per-component constitutive error (~65) was already close to 100% relative to the true RMS stress (~74) — meaning there is likely at least one more real issue beyond just the scale. **Watch the `constitutive` value in the first few thousand iterations**: if it still shows no downward trend compared to attempt 1's own printed log, interrupt and report rather than letting the full 50,000 iterations run on a loss that still isn't learning.

In [ ]:
# =====================================================================
#  CELL -- B3 mixed displacement-stress: REAL training run (50,000
#  iterations, production model size), then evaluation against the
#  held-out FEM dataset. See the markdown cell above for why this is
#  cleared to run now.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys
import time

_started = time.time()


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
if os.path.isdir('/content/drive/MyDrive'):
    print('Drive already mounted at /content/drive.')
else:
    drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
assert torch.cuda.is_available(), 'this cell needs a real GPU'
print('GPU:', torch.cuda.get_device_name(0))

import gc
for _attr in ('last_traceback', 'last_value', 'last_type'):
    if hasattr(sys, _attr):
        delattr(sys, _attr)
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory at start: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
      f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved (should be ~0 -- if not, '
      'use Runtime > Restart session before trusting this run)')

from omar_pfem.train_B3_mixed_stress import get_args, train
from omar_pfem.data.data_generate_B3_dataset import DEFAULT_RESOLUTION

device = torch.device('cuda')

R = '/content/drive/MyDrive/pfem_run'
OUTPUT_DIR = f'{R}/b3_mixed_stress_training_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = get_args([
    '--n_iters', '50000',
    '--batch_size', '8',
    '--log_every', '500',
    '--ckpt_every', '5000',
    '--output_dir', OUTPUT_DIR,
    # w_equilibrium=28000 and p_output_scale=500.0 are this script's own
    # defaults, confirmed stable at production scale by the diagnostic
    # notebook -- not overridden here.
])
n_elem = (DEFAULT_RESOLUTION[0] - 1) * (DEFAULT_RESOLUTION[1] - 1) * (DEFAULT_RESOLUTION[2] - 1)
print(f'\nTraining at {DEFAULT_RESOLUTION} -> {n_elem} elements, {args.n_iters} iterations, '
      f'batch_size={args.batch_size}, w_constitutive={args.w_constitutive}, '
      f'w_equilibrium={args.w_equilibrium}, p_output_scale={args.p_output_scale}, '
      f'checkpoints -> {OUTPUT_DIR}')
print('Watch the s/iteration rate in the first few hundred iterations -- if it is wildly '
      'different from what the production diagnostic implied, interrupt and report the '
      'observed rate rather than letting a possibly-broken run continue for hours.')

torch.cuda.reset_peak_memory_stats(device)
t0 = time.time()
model = train(args, device)
elapsed_total = time.time() - t0

print(f'\n{"=" * 90}')
print(f'Training finished in {elapsed_total:.1f}s ({elapsed_total / args.n_iters:.4f}s/iteration average).')
print(f'GPU peak memory during training: '
      f'{torch.cuda.max_memory_allocated(device) / 1e6:.1f}MB allocated, '
      f'{torch.cuda.max_memory_reserved(device) / 1e6:.1f}MB reserved.')

# =====================================================================
#  Evaluation: same held-out FEM dataset the baseline DEM checkpoint was
#  evaluated on, so the two sets of numbers are directly comparable.
# =====================================================================
print(f'\n{"=" * 90}')
print('=== Evaluating the final checkpoint against the held-out FEM dataset ===')
CHECKPOINT = f'{OUTPUT_DIR}/checkpoint_{args.n_iters}.pt'
DATASET = f'{R}/b3_dataset_clean_holdout/dataset.h5'
OUT_JSON = f'{OUTPUT_DIR}/qois_{args.n_iters}.json'

sys.argv = [
    'evaluate_B3_mixed_stress.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3_mixed_stress import main as evaluate_main
evaluate_main()

try:
    from omar_pfem.run_manifest import write_manifest
    write_manifest(OUTPUT_DIR, kind='b3_mixed_stress_training',
                    args={'n_iters': args.n_iters, 'batch_size': args.batch_size,
                          'resolution': DEFAULT_RESOLUTION, 'lr': args.lr,
                          'w_constitutive': args.w_constitutive,
                          'w_equilibrium': args.w_equilibrium,
                          'p_output_scale': args.p_output_scale},
                    started_at=_started,
                    results={'elapsed_total_s': elapsed_total,
                             's_per_iter': elapsed_total / args.n_iters},
                    outputs=[CHECKPOINT, OUT_JSON],
                    notes="First real training run of the mixed displacement-stress "
                          "formulation (Professor Rabczuk's item 1, 2026-10-16/18/20 "
                          "emails) -- network predicts u and P jointly, trained on "
                          "constitutive consistency + equilibrium residuals instead of "
                          "total potential energy. Preceded by 5 CPU correctness checks "
                          "and a production-scale fixed-batch diagnostic (both passing) "
                          "before any real training time was spent. See "
                          "evaluate_B3_mixed_stress.py's own output/qois_*.json for "
                          "whether the direct stress prediction beats the implicit "
                          "(autograd-from-u) one against the held-out FEM dataset.")
except Exception as e:
    print(f'[manifest] not recorded: {e}')

print('\nDone. Send this cell\'s full printed output back, especially the "Key comparison '
      'for Professor Rabczuk\'s item 1" line near the end of the evaluation output.')
